In [ ]:
# imports
import os
from dotenv import load_dotenv
import requests
from IPython.display import Markdown, display
from bs4 import BeautifulSoup
from openai import OpenAI
# If you get an error running this cell, then please head over to the troubleshooting notebook!

In [ ]:
#if we would have used gpt/frontier models, then this step would have came 

#load_dotenv(override=True)
#gpt_api_key = os.getenv("OPENAI_API_KEY")


In [ ]:
openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

In [ ]:
# A class to represent a Webpage
# If you're not familiar with Classes, check out the "Intermediate Python" notebook

# Some websites need you to use proper headers when fetching them:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:

    def __init__(self, url):
        """
        Create this Website object from the given url using the BeautifulSoup library
        """
        self.url = url
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)

In [ ]:
system_prompt = "You are a financial expert and you have to provide a summary and suggestions on" \
"what can the market look like today. Respond in Markdown"

In [ ]:
def promptsFor(website):
    user_prompt = f"you are looking at a website called {website.title}"
    user_prompt+= "\nThe contents of this website is as follows; \
please provide a short summary of this website in markdown. \
If it includes news or announcements, then summarize these too.\n\n"
    user_prompt+= website.text
    print(user_prompt)
    return user_prompt

In [ ]:
def messagesFor(website):
    return [
        {'role':'system', 'content':system_prompt},
        {'role':'user', 'content':promptsFor(website)}
    ]

In [ ]:
def summarize(url):
    scraped = Website(url)
    messages = messagesFor(scraped)
    resp = openai.chat.completions.create(
        model='llama3.2',
        messages=messages
    )
    

In [ ]:
def display_summary(summary):    
    display(Markdown(summary))

In [ ]:
url = "https://coinmarketcap.com/currencies/bitcoin/historical-data/3"
summary = summarize(url)
display_summary(summary)